## Load libraries and set project directory

In [1]:
# Load libraries
import pandas as pd
import gseapy as gp
from gseapy import barplot, dotplot, heatmap
from gseapy import Msigdb
import matplotlib.pyplot as plt
import os
import numpy as np
import statsmodels.stats.multitest
#.stats.multitest.fdrcorrection(pvals, alpha=0.05, method='indep', is_sorted=False)[source]
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.weightstats import ttest_ind
import networkx as nx
import random
from sklearn import linear_model
from sklearn import linear_model
import scipy
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.cluster.hierarchy import cophenet
from scipy.spatial.distance import pdist
%matplotlib inline

# Set project directory as working directory
project_dir = "/lab-share/Pulmonary-Chun-e2/Public/data/czi_integration"
os.chdir(project_dir)

# Prep data

## Read in differential gene list

The differential genes are comparing in NPC2 vs Control samples for both AT2 cells and macrophage cells.

In [2]:
# Read in data
de_genes_at2 = pd.read_csv("output/08_de_at2_vs_ctrl.csv")
de_genes_macs = pd.read_csv("output/08_de_macs_vs_ctrl.csv")

de_genes_at2.columns.values[0] = "gene_symbol"
de_genes_macs.columns.values[0] = "gene_symbol"

print(de_genes_at2.head())
print(de_genes_macs.head())

  gene_symbol          p_val  avg_log2FC  pct.1  pct.2      p_val_adj
0       DTHD1  8.825132e-278   -3.656918  0.306  0.104  1.544398e-274
1      LMNTD1  1.879619e-219   -1.045546  0.427  0.192  3.289333e-216
2    KIAA1211  3.240984e-183   -8.585989  0.622  0.333  5.671722e-180
3       CHST9  1.870905e-166   -4.316990  0.415  0.212  3.274084e-163
4    CABCOCO1  2.251106e-158   -0.104351  0.335  0.157  3.939435e-155
  gene_symbol          p_val  avg_log2FC  pct.1  pct.2      p_val_adj
0   MTRNR2L12   0.000000e+00    4.192338  0.756  0.088   0.000000e+00
1        XIST   0.000000e+00  -12.668980  0.000  0.642   0.000000e+00
2      HS3ST2   0.000000e+00    5.056330  0.514  0.014   0.000000e+00
3      TCF7L2  1.615220e-264   -3.323688  0.319  0.792  5.066784e-260
4      FNDC3B  1.311863e-256   -1.922318  0.712  0.932  4.115185e-252


## Filter data

In [3]:
# Process data
de_genes_at2 = de_genes_at2[de_genes_at2["gene_symbol"] != "-"]  # remove genes with missing symbols
de_genes_at2 = de_genes_at2[de_genes_at2["p_val_adj"].notna()] # remove genes with missing pvalues

de_genes_macs = de_genes_macs[de_genes_macs["gene_symbol"] != "-"]  # remove genes with missing symbols
de_genes_macs = de_genes_macs[de_genes_macs["p_val_adj"].notna()] # remove genes with missing pvalues

# Filter to DE genes with a adjusted p-value < 0.05 and 0.01
## AT2
padj01de_genes_at2 = de_genes_at2[de_genes_at2["p_val_adj"] < 0.1]
padj005de_genes_at2 = de_genes_at2[de_genes_at2["p_val_adj"] < 0.05]

print(len(padj01de_genes_at2))
print(len(padj005de_genes_at2))

## Macrophages
padj01de_genes_macs = de_genes_macs[de_genes_macs["p_val_adj"] < 0.1]
padj005de_genes_macs = de_genes_macs[de_genes_macs["p_val_adj"] < 0.05]

print(len(padj01de_genes_macs))
print(len(padj005de_genes_macs))

922
895
4993
4808


## Rank genes by log2 fold change

You need to separate by positive and negative because positive log2fc gets a positive score and vice versa and then the two separated rankings get combined.
The bigger the absolute value of the ranking, the bigger absolute value fo the log2fc.

### AT2 Cells

In [4]:
# Sort by log2fc
de_genes_at2 = de_genes_at2.sort_values("avg_log2FC", ascending = False)

# Split into positive and negative log2fc, rank accordingly
num_pos_genes_at2 = len(de_genes_at2[de_genes_at2["avg_log2FC"] >= 0])
num_list_positive_log2fc_at2 = list(range(num_pos_genes_at2, 0, -1))
num_neg_genes_at2 = len(de_genes_at2[de_genes_at2["avg_log2FC"] < 0])
num_list_negative_log2fc_at2 = list(range(-1, -1 * (num_neg_genes_at2 + 1), -1))

# Add ranks to dataframe
log2fc_ranks = num_list_positive_log2fc_at2 + num_list_negative_log2fc_at2
de_genes_at2["log2fc_ranks"] = log2fc_ranks

### Macrophage Cells

In [5]:
# Sort by log2fc
de_genes_macs = de_genes_macs.sort_values("avg_log2FC", ascending = False)

# Split into positive and negative log2fc, rank accordingly
num_pos_genes_macs = len(de_genes_macs[de_genes_macs["avg_log2FC"] >= 0])
num_list_positive_log2fc_macs = list(range(num_pos_genes_macs, 0, -1))
num_neg_genes_macs = len(de_genes_macs[de_genes_macs["avg_log2FC"] < 0])
num_list_negative_log2fc_macs = list(range(-1, -1 * (num_neg_genes_macs + 1), -1))

# Add ranks to dataframe
log2fc_ranks = num_list_positive_log2fc_macs + num_list_negative_log2fc_macs
de_genes_macs["log2fc_ranks"] = log2fc_ranks

## Rank genes by p-value

Again you have to separate by positive and negative log2fc for the same reasons as above.

You have to use regular p-value instead of adjusted p-value because the adjusted p-value will give less ties; this is especially important for Wilcoxon rank sum tests.

### AT2 Cells

In [6]:
# Transform p-values into -log(pvalues) for easier sorting (more significant genes will have bigger numbers)
minus_log_pval_at2 = list(-np.log10(de_genes_at2["p_val"].tolist()))
de_genes_at2["minus_log10_p_val"] = minus_log_pval_at2

# Sort by p-value
de_genes_at2 = de_genes_at2.sort_values("minus_log10_p_val", ascending = False)

Now that the genes are sorted, we can split them into positive and negative, rank, and then recombine the rankings.

In [7]:
# Split into positive and negative log2fc and rank according to -log(pvalue)
pos_log2fc_at2 = de_genes_at2[de_genes_at2["avg_log2FC"] >= 0]
pos_log2fc_at2["pval_ranks"] = num_list_positive_log2fc_at2 # List already defined above in log2fc ranking step


neg_log2fc_at2 = de_genes_at2[de_genes_at2["avg_log2FC"] < 0]
neg_log2fc_at2 = neg_log2fc_at2.sort_values("minus_log10_p_val", ascending = True)
neg_log2fc_at2["pval_ranks"] = num_list_negative_log2fc_at2

# Add ranks to dataframe
de_genes_at2 = pd.concat([pos_log2fc_at2, neg_log2fc_at2], axis=0)
print(de_genes_at2.head())
print(de_genes_at2.tail())

   gene_symbol          p_val  avg_log2FC  pct.1  pct.2      p_val_adj  \
5       CFAP47  8.207227e-149    3.417491  0.416  0.228  1.436265e-145   
10      SEMA6A  5.286429e-133    0.683170  0.035  0.062  9.251251e-130   
15        SYN3  4.387201e-123    2.933876  0.216  0.070  7.677601e-120   
19        TBX3  3.796610e-116    3.032769  0.037  0.043  6.644068e-113   
20       TTC29  4.326269e-115    1.325708  0.236  0.130  7.570971e-112   

    log2fc_ranks  minus_log10_p_val  pval_ranks  
5            285         148.085804         412  
10            85         132.276838         411  
15           257         122.357813         410  
19           263         115.420604         409  
20           145         114.363886         408  
  gene_symbol          p_val  avg_log2FC  pct.1  pct.2      p_val_adj  \
4    CABCOCO1  2.251106e-158   -0.104351  0.335  0.157  3.939435e-155   
3       CHST9  1.870905e-166   -4.316990  0.415  0.212  3.274084e-163   
2    KIAA1211  3.240984e-183   -8.58

/tmp/ipykernel_1053840/1875178422.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pos_log2fc_at2["pval_ranks"] = num_list_positive_log2fc_at2 # List already defined above in log2fc ranking step


### Macrophage Cells

In [8]:
# Transform p-values into -log(pvalues) for easier sorting (more significant genes will have bigger numbers)
minus_log_pval_macs = list(-np.log10(de_genes_macs["p_val"].tolist()))
de_genes_macs["minus_log10_p_val"] = minus_log_pval_macs

# Sort by p-value
de_genes_macs = de_genes_macs.sort_values("minus_log10_p_val", ascending = False)

/tmp/ipykernel_1053840/2302320460.py:2: RuntimeWarning: divide by zero encountered in log10
  minus_log_pval_macs = list(-np.log10(de_genes_macs["p_val"].tolist()))


In [9]:
# Split into positive and negative log2fc and rank according to -log(pvalue)
pos_log2fc_macs = de_genes_macs[de_genes_macs["avg_log2FC"] >= 0]
pos_log2fc_macs["pval_ranks"] = num_list_positive_log2fc_macs # List already defined above in log2fc ranking step

neg_log2fc_macs = de_genes_macs[de_genes_macs["avg_log2FC"] < 0]
neg_log2fc_macs = neg_log2fc_macs.sort_values("minus_log10_p_val", ascending = True)
neg_log2fc_macs["pval_ranks"] = num_list_negative_log2fc_macs

# Add ranks to dataframe
de_genes_macs = pd.concat([pos_log2fc_macs, neg_log2fc_macs], axis=0)
print(de_genes_macs.head())
print(de_genes_macs.tail())

   gene_symbol          p_val  avg_log2FC  pct.1  pct.2      p_val_adj  \
2       HS3ST2   0.000000e+00    5.056330  0.514  0.014   0.000000e+00   
0    MTRNR2L12   0.000000e+00    4.192338  0.756  0.088   0.000000e+00   
7    LINC01500  3.373849e-210    5.547856  0.341  0.011  1.058343e-205   
8        HMCN1  2.897842e-206    2.356621  0.571  0.130  9.090239e-202   
11         NTM  6.847481e-199    1.784364  0.500  0.092  2.147986e-194   

    log2fc_ranks  minus_log10_p_val  pval_ranks  
2           2063                inf        2089  
0           2039                inf        2088  
7           2071         209.471874        2087  
8           1942         205.537925        2086  
11          1864         198.164469        2085  
  gene_symbol          p_val  avg_log2FC  pct.1  pct.2      p_val_adj  \
6       LSAMP  4.526711e-221   -5.721042  0.166  0.635  1.419984e-216   
5        SVIL  1.006716e-250   -3.010149  0.320  0.780  3.157967e-246   
4      FNDC3B  1.311863e-256   -1.92

/tmp/ipykernel_1053840/15624781.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pos_log2fc_macs["pval_ranks"] = num_list_positive_log2fc_macs # List already defined above in log2fc ranking step


# Run GSEA

In [10]:
# MSIGDB (molecular signature database)
msig = Msigdb()

# list msigdb version you wanna query
msig.list_dbver()

# list categories given most recent dbver.
msig.list_category(dbver = "2025.1.Hs")

['c1.all',
 'c2.all',
 'c2.cgp',
 'c2.cp.biocarta',
 'c2.cp.kegg_legacy',
 'c2.cp.kegg_medicus',
 'c2.cp.pid',
 'c2.cp.reactome',
 'c2.cp',
 'c2.cp.wikipathways',
 'c3.all',
 'c3.mir.mir_legacy',
 'c3.mir.mirdb',
 'c3.mir',
 'c3.tft.gtrd',
 'c3.tft.tft_legacy',
 'c3.tft',
 'c4.3ca',
 'c4.all',
 'c4.cgn',
 'c4.cm',
 'c5.all',
 'c5.go.bp',
 'c5.go.cc',
 'c5.go.mf',
 'c5.go',
 'c5.hpo',
 'c6.all',
 'c7.all',
 'c7.immunesigdb',
 'c7.vax',
 'c8.all',
 'h.all',
 'msigdb']

In [11]:
# Variables to loop through
cell_types = ["at2", "macrophage"]
databases = ["hallmark", "kegg", "pid"]
rankings = ["log2fc", "pval"]

In [13]:
for cell in cell_types:
    print(cell)
    for database in databases:
        print(database)
        for ranking in rankings:
            print(ranking)
            # Prerank
            if cell == "at2":
                prerank_list = de_genes_at2.sort_values(f"{ranking}_ranks", ascending = False)
            elif cell == "macrophage":
                prerank_list = de_genes_macs.sort_values(f"{ranking}_ranks", ascending = False)
            else:
                print(f"Need to provide differential expression for selected cell type: {cell}")
            prerank_list[["gene_symbol", f"{ranking}_ranks"]]
            # Select pathway database reference
            if database == "hallmark":
                gmt = msig.get_gmt(category = "h.all", dbver = "2025.1.Hs")
            elif database == "kegg":
                gmt = msig.get_gmt(category = "c2.cp.kegg_legacy", dbver = "2025.1.Hs")
            elif database == "pid":
                gmt = msig.get_gmt(category = "c2.cp.pid", dbver = "2025.1.Hs")
            else:
                print("Error with pathway database selection")
                                   
            # Run GSEA
            pre_res = gp.prerank(rnk = prerank_list[["gene_symbol", f"{ranking}_ranks"]], 
                                 gene_sets = gmt,
                                 threads = 4,
                                 min_size = 10,
                                 max_size = 500,
                                 permutation_num = 1000, # reduce number to speed up testing
                                 outdir = f"{project_dir}/output/09_gsea/{database}_{cell}_{ranking}_ranking", # don't write to disk
                                 graph_num = 25,
                                 format = "png",
                                 verbose=True, # see what's going on behind the scenes
                                )
            # Preview results
            print(pre_res.res2d.head(5))
            # Swoosh plot
            terms = pre_res.res2d.Term
            axs = pre_res.plot(terms=terms[0:5],
                               #legend_kws={'loc': (1.2, 0)}, # set the legend loc
                               show_ranking=True, # whether to show the second yaxis
                               figsize=(3,4)
                              )
            print(axs)
            # Dot plot
            ax = dotplot(pre_res.res2d,
                         column="FDR q-val",
                         title=f"{database} Reference for {cell} Cells\n {ranking} Ranked",
                         cmap=plt.cm.viridis,
                         size=6, # adjust dot size
                         figsize=(4,6), cutoff=1, show_ring=False)
            print(ax)


at2
hallmark
log2fc


2025-07-17 13:46:53,526 [INFO] Parsing data files for GSEA.............................
2025-07-17 13:46:53,533 [INFO] 0017 gene_sets have been filtered out when max_size=500 and min_size=10
2025-07-17 13:46:53,534 [INFO] 0033 gene_sets used for further statistical testing.....
2025-07-17 13:46:53,534 [INFO] Start to run GSEA...Might take a while..................
2025-07-17 13:46:53,867 [INFO] Congratulations. GSEApy runs successfully................



      Name                            Term        ES       NES NOM p-val  \
0  prerank            HALLMARK_COAGULATION -0.522899 -1.433607  0.020202   
1  prerank  HALLMARK_FATTY_ACID_METABOLISM -0.561004 -1.392497  0.051742   
2  prerank           HALLMARK_ADIPOGENESIS -0.530579 -1.359988  0.065353   
3  prerank       HALLMARK_MTORC1_SIGNALING -0.522602 -1.237284  0.164882   
4  prerank      HALLMARK_KRAS_SIGNALING_DN  -0.48162 -1.230256  0.163412   

  FDR q-val FWER p-val  Tag %  Gene %  \
0  0.643196      0.472  16/25  36.77%   
1  0.509693      0.631   8/13  28.37%   
2  0.472471       0.74   9/16  30.93%   
3       1.0      0.975   8/11  36.18%   
4  0.897298       0.98   5/16   9.65%   

                                          Lead_genes  
0  DPP4;PRSS23;TIMP1;BMP1;ANXA1;ITGA2;APOC1;F3;CT...  
1  ACADL;HSP90AA1;PTPRG;BMPR1B;MAOA;MGLL;CD36;ALD...  
2    ACADL;PDCD4;PLIN2;MGLL;ABCA1;CD36;LIFR;C3;NABP1  
3       NIBAN1;BCAT1;CTSC;SLA;SLC2A3;RRM2;TFRC;SYTL2  
4                  SI

2025-07-17 13:46:57,279 [INFO] Parsing data files for GSEA.............................
2025-07-17 13:46:57,290 [INFO] 0017 gene_sets have been filtered out when max_size=500 and min_size=10
2025-07-17 13:46:57,290 [INFO] 0033 gene_sets used for further statistical testing.....
2025-07-17 13:46:57,291 [INFO] Start to run GSEA...Might take a while..................
2025-07-17 13:46:57,638 [INFO] Congratulations. GSEApy runs successfully................



      Name                           Term        ES       NES NOM p-val  \
0  prerank  HALLMARK_BILE_ACID_METABOLISM -0.575375 -1.385471   0.06538   
1  prerank     HALLMARK_KRAS_SIGNALING_DN  -0.52437 -1.343515  0.065625   
2  prerank          HALLMARK_ANGIOGENESIS -0.524161 -1.286071  0.126702   
3  prerank        HALLMARK_G2M_CHECKPOINT -0.539139 -1.259191   0.15393   
4  prerank             HALLMARK_APOPTOSIS -0.444555  -1.22374  0.143001   

  FDR q-val FWER p-val  Tag %  Gene %  \
0       1.0      0.676   5/11  19.04%   
1  0.845347      0.808   7/16  23.11%   
2  0.956934       0.92   7/13  29.28%   
3  0.883094      0.951   7/10  43.01%   
4  0.929385      0.977  15/27  36.18%   

                                          Lead_genes  
0                    ABCA1;ALDH1A1;ABCA6;ABCA3;ABCA9  
1        BMPR1B;SGK1;SIDT1;EDN1;SLC16A7;PTGFR;CAMK1D  
2          JAG1;TIMP1;FSTL1;VCAN;TNFRSF21;LPL;COL3A1  
3          SLC38A1;MT2A;EZH2;TOP2A;MEIS2;HIF1A;EFNA5  
4  PLAT;BIRC3;TIMP3;SLC20A1

...

2025-07-17 13:47:02,036 [INFO] Parsing data files for GSEA.............................
2025-07-17 13:47:02,040 [INFO] 0164 gene_sets have been filtered out when max_size=500 and min_size=10
2025-07-17 13:47:02,041 [INFO] 0032 gene_sets used for further statistical testing.....
2025-07-17 13:47:02,041 [INFO] Start to run GSEA...Might take a while..................
2025-07-17 13:47:02,587 [INFO] Congratulations. GSEApy runs successfully................



      Name                          Term        ES       NES NOM p-val  \
0  prerank           PID_INSULIN_PATHWAY  0.468254  1.556515  0.031746   
1  prerank  PID_ERBB1_DOWNSTREAM_PATHWAY  0.319417  1.353755  0.030303   
2  prerank         PID_INTEGRIN3_PATHWAY  -0.48738 -1.219132  0.181631   
3  prerank        PID_SYNDECAN_1_PATHWAY  0.290258  1.208167  0.162791   
4  prerank               PID_FGF_PATHWAY -0.485421 -1.133926  0.296692   

  FDR q-val FWER p-val  Tag %  Gene %  \
0  0.226358      0.047  11/11  53.51%   
1  0.305369       0.13  14/14  68.35%   
2       1.0      0.977   8/14  35.06%   
3  0.408582      0.234  14/14  71.24%   
4       1.0      0.997   9/10  50.82%   

                                          Lead_genes  
0  IRS1;SORBS1;NCK2;GRB10;EXOC6;SGK1;PTPN1;FOXO3;...  
1  EGR1;DUSP6;MAP3K1;FOS;DIAPH3;PIK3CD;PRKCA;GRB2...  
2       KDR;EDIL3;PLAUR;TNC;PDGFRB;COL1A1;PDGFB;FBN1  
3  COL5A2;COL4A4;COL4A3;COL5A1;COL8A1;COL15A1;COL...  
4  RUNX2;FGFR2;NCAM1;PLAUR;SDC2;M

2025-07-17 13:47:03,768 [INFO] Parsing data files for GSEA.............................
2025-07-17 13:47:03,772 [INFO] 0164 gene_sets have been filtered out when max_size=500 and min_size=10
2025-07-17 13:47:03,772 [INFO] 0032 gene_sets used for further statistical testing.....
2025-07-17 13:47:03,773 [INFO] Start to run GSEA...Might take a while..................
2025-07-17 13:47:03,985 [INFO] Congratulations. GSEApy runs successfully................



      Name                          Term        ES       NES NOM p-val  \
0  prerank               PID_TCR_PATHWAY  0.353408  1.292582  0.089286   
1  prerank           PID_CD8_TCR_PATHWAY  0.338187  1.193638  0.163265   
2  prerank        PID_SYNDECAN_1_PATHWAY -0.458243 -1.144594  0.295407   
3  prerank  PID_BETA_CATENIN_NUC_PATHWAY -0.447076   -1.1279  0.306653   
4  prerank  PID_ARF6_TRAFFICKING_PATHWAY -0.467823  -1.09895  0.338028   

  FDR q-val FWER p-val  Tag %  Gene %  \
0  0.724808      0.169  12/12  64.94%   
1  0.581615      0.256  12/12  66.45%   
2       1.0      0.994   3/14  11.82%   
3       1.0      0.995   6/14  31.58%   
4       1.0      0.997   2/10  17.40%   

                                          Lead_genes  
0  PAG1;CD247;PRKCE;CD86;HLA-DRA;MAP3K8;PRKCQ;GAB...  
1  PAG1;CD247;PRKCE;CD86;B2M;MAP3K8;PRKCQ;GRB2;LC...  
2                             COL12A1;COL6A2;COL14A1  
3                KCNIP4;TBL1X;VCAN;CAMK4;TLE4;MT-CO2  
4                                

2025-07-17 13:47:05,249 [INFO] Parsing data files for GSEA.............................
2025-07-17 13:47:05,269 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=10
2025-07-17 13:47:05,270 [INFO] 0050 gene_sets used for further statistical testing.....
2025-07-17 13:47:05,271 [INFO] Start to run GSEA...Might take a while..................
